# Train 10-K → Price Direction Model (SageMaker)

Self-contained training notebook. Upload this `.ipynb` to a SageMaker Studio or Classic notebook instance (recommended: **ml.g4dn.xlarge** — has one T4 GPU, ~$0.74/hr, plenty for FinBERT on ~200k sentences).

**Inputs** — the sentence-level CSVs from your Drive dataset, uploaded to S3:
```
s3://<bucket>/training/small_train.csv
s3://<bucket>/training/small_validate.csv
s3://<bucket>/training/small_test.csv
```

**Outputs** — written to `s3://<bucket>/models/xgb/v1/`:
- `booster.json` — XGBoost model
- `calibrator.pkl` — isotonic calibration wrapper
- `feature_cols.json` — canonical feature ordering (also consumed by the predict Lambda)
- `metrics.json` — validation AUC, test AUC, dataset sizes

**Pipeline in this notebook**:
1. Load + clean sentence-level CSVs
2. Score every sentence with FinBERT (GPU, checkpointed to parquet)
3. Tag each sentence with business aspects (keyword + section priors)
4. Aggregate sentence sentiment → per-filing aspect vector
5. Add YoY delta features (this filing minus prior filing per ticker)
6. Train XGBoost with early stopping, calibrate with isotonic regression on val
7. Evaluate on test, save artifacts to S3

## 1. Setup

In [ ]:
!pip install -q transformers==4.41.2 xgboost==2.0.3 scikit-learn==1.5.0 pandas==2.2.2 pyarrow==16.1.0 boto3==1.34.110 tqdm

In [ ]:
import json
import os
import pickle
import re
from dataclasses import dataclass
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

Edit the three variables below. `S3_BUCKET` is the bucket holding your CSVs and where artifacts will land. The version tag lets you retrain without overwriting.

In [ ]:
S3_BUCKET = "your-bucket-name"              # <-- CHANGE ME
DATA_PREFIX = "training/"                   # S3 prefix holding the CSVs
MODEL_VERSION = "v1"
MODEL_PREFIX = f"models/xgb/{MODEL_VERSION}/"

LOCAL_DATA = Path("./data"); LOCAL_DATA.mkdir(exist_ok=True)
LOCAL_CACHE = Path("./cache"); LOCAL_CACHE.mkdir(exist_ok=True)
LOCAL_MODEL = Path("./model"); LOCAL_MODEL.mkdir(exist_ok=True)

FINBERT_ID = "ProsusAI/finbert"
FINBERT_MAX_LEN = 256
FINBERT_BATCH = 64
FINBERT_LABELS = ("positive", "negative", "neutral")  # ProsusAI/finbert id2label order

HORIZON_LABEL = "label_30d"  # which horizon column we treat as y

s3 = boto3.client("s3")

## 3. Download CSVs from S3

In [ ]:
def pull_csv(name: str) -> pd.DataFrame:
    local = LOCAL_DATA / name
    if not local.exists():
        print(f"Downloading {name}...")
        s3.download_file(S3_BUCKET, f"{DATA_PREFIX}{name}", str(local))
    return pd.read_csv(local)

train_df = pull_csv("small_train.csv")
val_df = pull_csv("small_validate.csv")
test_df = pull_csv("small_test.csv")

print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")
train_df.head(3)

## 4. Clean

Same cleaning as the EDA notebook: drop duplicates, parse dates, normalize labels.

In [ ]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates().reset_index(drop=True).copy()
    df = df.dropna(subset=["sentence"])
    df["sentence"] = df["sentence"].str.strip()
    df = df[df["sentence"] != ""]
    df["filingDate"] = pd.to_datetime(df["filingDate"], errors="coerce")
    for col in ("label_1d", "label_5d", "label_30d"):
        df[col] = df[col].str.strip().str.lower()
    return df.reset_index(drop=True)

train_df = clean(train_df)
val_df = clean(val_df)
test_df = clean(test_df)
print(f"After clean — Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
print(train_df[HORIZON_LABEL].value_counts(normalize=True))

## 5. Aspect taxonomy

Same keyword-based tagger as `common/aspects.py` in the repo. A sentence can map to multiple aspects. If no keyword hits, fall back to the section prior (e.g. Item 1A → `risk_factors`).

In [ ]:
ASPECTS = (
    "revenue", "cash_flow", "margins", "ebitda",
    "future_plans", "risk_factors", "guidance",
)

KEYWORDS = {
    "revenue": ("revenue", "revenues", "net sales", "total sales", "top line",
                "net revenue", "product sales", "service revenue", "billings"),
    "cash_flow": ("cash flow", "cash flows", "operating cash", "free cash flow",
                  "cash from operations", "cash and cash equivalents",
                  "working capital", "liquidity"),
    "margins": ("gross margin", "operating margin", "net margin", "profit margin",
                "margin expansion", "margin compression", "cost of goods sold",
                "cogs", "gross profit"),
    "ebitda": ("ebitda", "adjusted ebitda", "earnings before interest",
               "operating income", "operating profit", "operating earnings"),
    "future_plans": ("we plan", "we intend", "we expect to", "we will", "strategy",
                     "initiative", "roadmap", "expansion", "invest in", "planned",
                     "in the coming year", "next fiscal", "long-term", "future growth"),
    "risk_factors": ("risk", "risks", "uncertainty", "adverse", "material adverse",
                     "could harm", "may fail", "litigation", "regulatory", "volatility",
                     "downturn", "recession", "dependence on", "could impact"),
    "guidance": ("guidance", "outlook", "forecast", "projected", "anticipate",
                 "estimate", "target", "we believe", "expected to be", "forward-looking"),
}

PATTERNS = {
    a: re.compile(r"\b(" + "|".join(re.escape(k) for k in kws) + r")\b", re.IGNORECASE)
    for a, kws in KEYWORDS.items()
}

SECTION_PRIORS = {
    "section_1": ("revenue", "future_plans"),
    "section_1A": ("risk_factors",),
    "section_7": ("revenue", "margins", "ebitda", "cash_flow", "guidance", "future_plans"),
    "section_7A": ("risk_factors",),
}

def tag_aspects(sentence: str, section: str | None) -> list[str]:
    hits = [a for a, pat in PATTERNS.items() if pat.search(sentence)]
    if not hits and section in SECTION_PRIORS:
        hits = list(SECTION_PRIORS[section])
    return hits

# Quick sanity check
tag_aspects("Our operating margin expanded due to lower COGS.", "section_7")

## 6. Score every sentence with FinBERT

This is the expensive step (~10-15 min for 200k sentences on a T4). Results are cached to parquet so a re-run of the notebook skips it.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(FINBERT_ID)
finbert = AutoModelForSequenceClassification.from_pretrained(FINBERT_ID).to(DEVICE).eval()
# Confirm label order — ProsusAI/finbert is [positive, negative, neutral] but double-check.
print(finbert.config.id2label)

In [ ]:
@torch.no_grad()
def finbert_score(sentences: list[str]) -> np.ndarray:
    """Return Nx3 array of [positive, negative, neutral] probabilities."""
    out = np.zeros((len(sentences), 3), dtype=np.float32)
    # Remap to canonical (positive, negative, neutral) order using the config.
    label_to_col = {"positive": 0, "negative": 1, "neutral": 2}
    model_cols = [label_to_col[finbert.config.id2label[i].lower()] for i in range(3)]
    for i in tqdm(range(0, len(sentences), FINBERT_BATCH), desc="FinBERT"):
        chunk = sentences[i : i + FINBERT_BATCH]
        enc = tokenizer(chunk, padding=True, truncation=True,
                        max_length=FINBERT_MAX_LEN, return_tensors="pt").to(DEVICE)
        probs = torch.softmax(finbert(**enc).logits, dim=-1).cpu().numpy()
        for j, c in enumerate(model_cols):
            out[i : i + len(chunk), c] = probs[:, j]
    return out

def score_split(name: str, df: pd.DataFrame) -> pd.DataFrame:
    cache = LOCAL_CACHE / f"{name}_finbert.parquet"
    if cache.exists():
        print(f"[cache hit] {cache}")
        return pd.read_parquet(cache)
    arr = finbert_score(df["sentence"].tolist())
    scored = df.copy()
    scored["p_positive"] = arr[:, 0]
    scored["p_negative"] = arr[:, 1]
    scored["p_neutral"] = arr[:, 2]
    scored.to_parquet(cache, index=False)
    return scored

train_scored = score_split("train", train_df)
val_scored = score_split("val", val_df)
test_scored = score_split("test", test_df)
train_scored[["sentence", "section", "p_positive", "p_negative", "p_neutral"]].head()

## 7. Aggregate to filing-level aspect vector

Per filing (keyed by `(ticker, filingDate)`): for each aspect, take the mean of `p_positive - p_negative` across sentences tagged with that aspect. Missing aspects → 0. Also emit counts and metadata.

In [ ]:
def build_filing_features(scored: pd.DataFrame) -> pd.DataFrame:
    scored = scored.copy()
    scored["signed"] = scored["p_positive"] - scored["p_negative"]
    rows = []
    for (ticker, fdate), grp in tqdm(scored.groupby(["tickers", "filingDate"], sort=False),
                                      desc="Aggregate"):
        sums = {a: 0.0 for a in ASPECTS}
        counts = {a: 0 for a in ASPECTS}
        for sent, section, signed in zip(grp["sentence"], grp["section"], grp["signed"]):
            for a in tag_aspects(sent, section):
                sums[a] += signed
                counts[a] += 1
        row = {"ticker": ticker, "filing_date": fdate}
        for a in ASPECTS:
            row[f"{a}_score"] = (sums[a] / counts[a]) if counts[a] else 0.0
            row[f"{a}_count"] = counts[a]
        row["y"] = int((grp[HORIZON_LABEL] == "positive").mean() >= 0.5)
        row["sic"] = int(grp["sic"].iloc[0])
        row["n_sentences"] = len(grp)
        rows.append(row)
    return pd.DataFrame(rows)

train_feat = build_filing_features(train_scored)
val_feat = build_filing_features(val_scored)
test_feat = build_filing_features(test_scored)
print(f"Train filings: {len(train_feat)}  Val filings: {len(val_feat)}  Test filings: {len(test_feat)}")
train_feat.head()

## 8. Add YoY delta features

For each ticker, this filing's aspect score minus the same ticker's prior filing score. Combines all splits first so val/test rows see their true prior even if that prior lives in the train split.

In [ ]:
def add_delta_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["ticker", "filing_date"]).reset_index(drop=True)
    score_cols = [f"{a}_score" for a in ASPECTS]
    prev = df.groupby("ticker")[score_cols].shift(1)
    for c in score_cols:
        df[c.replace("_score", "_delta")] = (df[c] - prev[c]).fillna(0.0)
    return df

combined = pd.concat(
    [train_feat.assign(_split="train"),
     val_feat.assign(_split="val"),
     test_feat.assign(_split="test")],
    ignore_index=True,
)
combined = add_delta_features(combined)
train_feat = combined[combined["_split"] == "train"].drop(columns="_split").reset_index(drop=True)
val_feat = combined[combined["_split"] == "val"].drop(columns="_split").reset_index(drop=True)
test_feat = combined[combined["_split"] == "test"].drop(columns="_split").reset_index(drop=True)

delta_cols = [f"{a}_delta" for a in ASPECTS]
train_feat[delta_cols].describe().round(3)

## 9. Feature matrix

Canonical column order lives in `FEATURE_COLS`. This list is saved with the artifacts and read by the predict Lambda so train/inference ordering cannot drift.

In [ ]:
FEATURE_COLS = []
for a in ASPECTS:
    FEATURE_COLS.extend([f"{a}_score", f"{a}_count", f"{a}_delta"])
FEATURE_COLS.extend(["sic", "n_sentences"])
print(f"{len(FEATURE_COLS)} features:", FEATURE_COLS)

X_train, y_train = train_feat[FEATURE_COLS].values, train_feat["y"].values
X_val, y_val = val_feat[FEATURE_COLS].values, val_feat["y"].values
X_test, y_test = test_feat[FEATURE_COLS].values, test_feat["y"].values
print(f"X_train: {X_train.shape}  y_train mean: {y_train.mean():.3f}")

## 10. Train XGBoost

In [ ]:
booster = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    eval_metric="logloss",
    early_stopping_rounds=25,
    tree_method="hist",
    random_state=42,
)
booster.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f"Best iteration: {booster.best_iteration}")

In [ ]:
# Feature importance (gain)
importance = booster.get_booster().get_score(importance_type="gain")
imp_df = pd.DataFrame({
    "feature": [FEATURE_COLS[int(f[1:])] for f in importance.keys()],
    "gain": list(importance.values()),
}).sort_values("gain", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(imp_df["feature"], imp_df["gain"], color="#3498db", edgecolor="black", linewidth=0.3)
ax.set_xlabel("Gain"); ax.set_title("XGBoost feature importance")
plt.tight_layout(); plt.show()

## 11. Calibrate on validation

Isotonic regression on the validation split turns raw XGBoost scores into calibrated probabilities — so a `0.70` prediction actually corresponds to ~70% observed hit rate. This is what the `CONF_LONG=0.70` gate in the trading signal Lambda relies on.

In [ ]:
calibrator = CalibratedClassifierCV(booster, method="isotonic", cv="prefit")
calibrator.fit(X_val, y_val)

raw_val = booster.predict_proba(X_val)[:, 1]
cal_val = calibrator.predict_proba(X_val)[:, 1]
raw_test = booster.predict_proba(X_test)[:, 1]
cal_test = calibrator.predict_proba(X_test)[:, 1]

print(f"Val   uncalibrated AUC: {roc_auc_score(y_val, raw_val):.4f}")
print(f"Val   calibrated   AUC: {roc_auc_score(y_val, cal_val):.4f}")
print(f"Test  uncalibrated AUC: {roc_auc_score(y_test, raw_test):.4f}")
print(f"Test  calibrated   AUC: {roc_auc_score(y_test, cal_test):.4f}")

## 12. Evaluate

In [ ]:
y_pred_test = (cal_test >= 0.5).astype(int)
print("=== Test classification report ===")
print(classification_report(y_test, y_pred_test, target_names=["down", "up"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay(confusion_matrix(y_val, (cal_val >= 0.5).astype(int)),
                        display_labels=["down", "up"]).plot(ax=axes[0], cmap="Blues")
axes[0].set_title("Validation")
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_test),
                        display_labels=["down", "up"]).plot(ax=axes[1], cmap="Blues")
axes[1].set_title("Test")
plt.tight_layout(); plt.show()

In [ ]:
# Reliability / calibration plot — probability bins vs. observed hit rate
bins = np.linspace(0, 1, 11)
centers = (bins[:-1] + bins[1:]) / 2

def bin_hit_rate(probs, y):
    idx = np.digitize(probs, bins) - 1
    idx = np.clip(idx, 0, len(centers) - 1)
    return [y[idx == i].mean() if (idx == i).any() else np.nan for i in range(len(centers))]

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
ax.plot(centers, bin_hit_rate(raw_test, y_test), "o-", label="raw XGBoost")
ax.plot(centers, bin_hit_rate(cal_test, y_test), "s-", label="isotonic calibrated")
ax.set_xlabel("Predicted P(up)"); ax.set_ylabel("Observed hit rate")
ax.set_title("Reliability — test set"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 13. Save artifacts locally + to S3

The predict Lambda pulls these from `s3://{S3_BUCKET}/{MODEL_PREFIX}` at cold start. The SageMaker real-time endpoint for XGBoost should be pointed at the same S3 prefix as its `model_data`.

In [ ]:
booster.get_booster().save_model(LOCAL_MODEL / "booster.json")
with open(LOCAL_MODEL / "calibrator.pkl", "wb") as f:
    pickle.dump(calibrator, f)
with open(LOCAL_MODEL / "feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f)
metrics = {
    "model_version": MODEL_VERSION,
    "horizon_label": HORIZON_LABEL,
    "val_auc_raw": float(roc_auc_score(y_val, raw_val)),
    "val_auc_calibrated": float(roc_auc_score(y_val, cal_val)),
    "test_auc_raw": float(roc_auc_score(y_test, raw_test)),
    "test_auc_calibrated": float(roc_auc_score(y_test, cal_test)),
    "n_train_filings": int(len(train_feat)),
    "n_val_filings": int(len(val_feat)),
    "n_test_filings": int(len(test_feat)),
    "feature_cols": FEATURE_COLS,
}
with open(LOCAL_MODEL / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

for name in ("booster.json", "calibrator.pkl", "feature_cols.json", "metrics.json"):
    s3.upload_file(str(LOCAL_MODEL / name), S3_BUCKET, f"{MODEL_PREFIX}{name}")
    print(f"uploaded s3://{S3_BUCKET}/{MODEL_PREFIX}{name}")

print("\n=== metrics.json ===")
print(json.dumps(metrics, indent=2))

## 14. Package for SageMaker endpoint

Wrap `booster.json`, `calibrator.pkl`, `feature_cols.json` into a `model.tar.gz` and include the inference script the endpoint will run. Upload to the same S3 prefix; use that URI as `ModelDataUrl` when creating the SageMaker model.

In [ ]:
import tarfile

INFERENCE_SCRIPT = '''
import json, pickle
from pathlib import Path
import numpy as np
import xgboost as xgb

def model_fn(model_dir):
    root = Path(model_dir)
    booster = xgb.XGBClassifier()
    booster.load_model(root / "booster.json")
    with open(root / "calibrator.pkl", "rb") as f:
        calibrator = pickle.load(f)
    with open(root / "feature_cols.json") as f:
        cols = json.load(f)
    return {"booster": booster, "calibrator": calibrator, "cols": cols}

def input_fn(body, content_type):
    if content_type != "application/json":
        raise ValueError(content_type)
    p = json.loads(body)
    return p["features"]

def predict_fn(features, bundle):
    cols = bundle["cols"]
    if isinstance(features, dict):
        X = np.array([[float(features[c]) for c in cols]])
    else:
        X = np.asarray(features, dtype=float)
    proba = bundle["calibrator"].predict_proba(X)[:, 1]
    return {"probability_up": proba.tolist()}

def output_fn(pred, accept):
    return json.dumps(pred), "application/json"
'''

code_dir = LOCAL_MODEL / "code"; code_dir.mkdir(exist_ok=True)
(code_dir / "inference.py").write_text(INFERENCE_SCRIPT)
(code_dir / "requirements.txt").write_text("xgboost==2.0.3\nscikit-learn==1.5.0\n")

tar_path = LOCAL_MODEL / "model.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    for f in ("booster.json", "calibrator.pkl", "feature_cols.json"):
        tar.add(LOCAL_MODEL / f, arcname=f)
    tar.add(code_dir, arcname="code")

s3.upload_file(str(tar_path), S3_BUCKET, f"{MODEL_PREFIX}model.tar.gz")
print(f"model.tar.gz -> s3://{S3_BUCKET}/{MODEL_PREFIX}model.tar.gz")

## 15. Deploy endpoint (optional — run once)

Creates a real-time SageMaker endpoint named `xgboost-endpoint`. Only run this cell if you're ready to incur endpoint costs (an `ml.t2.medium` is ~$0.06/hr, always-on).

In [ ]:
# Uncomment to deploy.
#
# import sagemaker
# from sagemaker.xgboost import XGBoostModel
#
# role = sagemaker.get_execution_role()
# model = XGBoostModel(
#     model_data=f"s3://{S3_BUCKET}/{MODEL_PREFIX}model.tar.gz",
#     role=role,
#     entry_point="inference.py",
#     framework_version="1.7-1",
# )
# predictor = model.deploy(
#     initial_instance_count=1,
#     instance_type="ml.t2.medium",
#     endpoint_name="xgboost-endpoint",
# )
# print("Endpoint:", predictor.endpoint_name)

## Next steps

1. **Deploy FinBERT endpoint** separately (it's a pretrained model, just needs a HuggingFace SageMaker container). Name it `finbert-endpoint` to match the predict Lambda's env var.
2. **Apply the CloudFormation stack** (`scripts/deploy.sh` in the repo) — creates S3, Aurora, DynamoDB, Lambdas, ECS Fargate for the dashboard.
3. **Seed the watchlist** (`scripts/seed_watchlist.py AAPL MSFT NVDA ...`).
4. **Smoke-test**: drop a real 10-K HTML into `s3://<ingest-bucket>/raw/AAPL/<accession>/primary.htm` and watch the flow through to the dashboard.
5. If the calibrated test AUC is below ~0.58, the baseline aspect features aren't enough — try longer context (tokenize at 512), a larger ABSA model, or add price-history features (prior N-day return) to the XGBoost input.